Produce ShopFlow orders to the orders topic, dafely

In [1]:
import random, sys
from collections import Counter

In [5]:
from confluent_kafka import Producer

In [6]:
%run order.ipynb

In [7]:
producer = Producer({"bootstrap.servers": "localhost:9092",
                     "acks": "all",
                     "enable.idempotence": True,
                     "linger.ms": 20,
                     "compression.type": "lz4"})

per_partition = Counter()

In [8]:
count = 1
first = 1

In [9]:
def on_delivery(err, msg):
    if err is not None:
        print("deliver failed.")
    else:
        per_partition[msg.partition()] += 1

In [11]:
rng = random.Random(first)

In [12]:
for n in range(first, first + count):
    order = make_order(n, rng)
    producer.produce("orders", key=order["customer_id"], value=encode(order), callback=on_delivery)
    producer.poll(0)

producer.flush()
print(f"sent {count} orders, per partition {dict(sorted(per_partition.items()))}")

sent 1 orders, per partition {2: 1}


Sample run:
> papermill producer.ipynb output_producer.ipynb -p first 300
# █

In [ ]:
# Check the offsets
# > kafka-get-offsets.sh --bootstrap-server $BS --topic orders

In [ ]:
# Check consumer
# > kafka-console-consumer.sh --bootstrap-server $BS --topic orders --from-beginning --max-messages 2 --formatter-property print.key=true